# Predicción de interacciones medicamento-medicamento

Modelo basado en una red neuronal artificial poco profunda (MLP) construido con Keras/TensorFlow. Recibe como entrada el vector de distancias químicas de un medicamento respecto a los 548 medicamentos del conjunto, y devuelve a la salida la probabilidad de interacción con cada uno de ellos. Como el problema es multietiqueta (un medicamento puede interactuar con varios), se usa activación sigmoide en la salida y pérdida `binary_crossentropy`. El entrenamiento utiliza el optimizador Adam con una tasa de aprendizaje baja porque la matriz de interacciones es dispersa.

## 1. Importar librerías

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

## 2. Cargar los datos

Se leen los tres archivos del dataset: la lista de medicamentos (relación drugBankID ↔ CompoundID), la matriz de distancias químicas y la matriz de interacciones.

In [ ]:
DATA_DIR = 'datasetDelLaboratorio10'

lista = pd.read_csv(os.path.join(DATA_DIR, 'lista_medicamentos.txt'), sep=r'\s+', header=None, names=['cid_num', 'drugBankID'])
lista['CompoundID'] = lista['cid_num'].apply(lambda n: f'CID{int(n):09d}')

distancias = pd.read_csv(os.path.join(DATA_DIR, 'matriz_de_distancias.csv'), index_col=0)
interacciones = pd.read_csv(os.path.join(DATA_DIR, 'matriz_interaciones_medicamento_medicamento.csv'), index_col=0)

compound_ids = list(distancias.columns)
distancias = distancias.loc[compound_ids, compound_ids]
interacciones = interacciones.loc[compound_ids, compound_ids]

print(f'Medicamentos: {len(compound_ids)}')
print(f'Distancias: {distancias.shape}')
print(f'Interacciones: {interacciones.shape}')

## 3. Preparar los datos para el modelo

Cada medicamento se representa por un vector de 548 distancias (una fila de la matriz de distancias). La etiqueta es la fila correspondiente de la matriz de interacciones (vector binario de tamaño 548).

In [ ]:
X = distancias.values.astype(np.float32)
Y = interacciones.values.astype(np.float32)

print(f'X shape: {X.shape}')
print(f'Y shape: {Y.shape}')
print(f'Densidad de interacciones: {Y.mean():.4f}')

## 4. Construir el modelo

Red neuronal poco profunda con tres capas ocultas y una capa de salida de tamaño 548 (una salida por medicamento). Las capas ocultas usan activación ReLU y la capa de salida activación sigmoide. Como pérdida se usa `binary_crossentropy` (apropiada para clasificación multietiqueta) y como optimizador Adam con una tasa de aprendizaje baja.

In [ ]:
tf.random.set_seed(42)
n = X.shape[1]

modelo = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(n,)),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(n, activation='sigmoid'),
])

modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)
modelo.summary()

## 5. Entrenar el modelo

In [ ]:
history = modelo.fit(X, Y, epochs=200, batch_size=32, verbose=0)
print(f"Loss final: {history.history['loss'][-1]:.4f}")
print(f"Accuracy final: {history.history['accuracy'][-1]:.4f}")

pd.DataFrame(history.history).plot(figsize=(8, 5), grid=True, xlabel='Epoch')
plt.title('Evolución del entrenamiento')
plt.show()

## 6. Función de predicción por drugBankID

Recibe un drugBankID y devuelve la precisión, recall, F1, AUC y los pares `(CompoundID, probabilidad de interacción)` para cada uno de los 548 medicamentos.

In [ ]:
UMBRAL = 0.5

def predecir_interacciones(drug_bank_id):
    fila = lista[lista['drugBankID'] == drug_bank_id]
    if fila.empty:
        raise ValueError(f"drugBankID '{drug_bank_id}' no encontrado.")
    compound_id = fila.iloc[0]['CompoundID']
    idx = compound_ids.index(compound_id)

    x = X[idx:idx + 1]
    y_real = Y[idx]
    y_prob = modelo.predict(x, verbose=0)[0]
    y_pred = (y_prob >= UMBRAL).astype(int)

    precision = precision_score(y_real, y_pred, zero_division=0)
    recall = recall_score(y_real, y_pred, zero_division=0)
    f1 = f1_score(y_real, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_real, y_prob)
    except ValueError:
        auc = float('nan')

    pares = pd.DataFrame({'CompoundID': compound_ids, 'probabilidad_interaccion': y_prob})
    return compound_id, precision, recall, f1, auc, pares

## 7. Ejemplo de uso

In [ ]:
drug_bank_id = 'DB00945'
compound_id, precision, recall, f1, auc, pares = predecir_interacciones(drug_bank_id)

print(f'drugBankID: {drug_bank_id}  CompoundID: {compound_id}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1:        {f1:.4f}')
print(f'AUC:       {auc:.4f}')
pares.head(10)

## Resumen

Se entrenó una red neuronal poco profunda con tres capas ocultas y salida `sigmoid` de tamaño 548 sobre la matriz de distancias químicas, usando como etiqueta la matriz de interacciones. El modelo entrega para cada medicamento (identificado por su drugBankID) la probabilidad de interacción con cada uno de los 548 medicamentos del conjunto, junto con las métricas de precisión, recall, F1 y AUC.